# NB02 - Captioning (the pairing backbone)

**Run once, after NB01.** Captions every real image with **Florence-2-large** (detailed mode), cleans and length-caps each caption deterministically, and stores them in `captions/*.parquet` keyed by `source_real_id`.

This table is what makes the six AI partners share one prompt - every generator reads **these exact captions**, so do not regenerate them per model.

### Prerequisites
- NB00 and NB01 done.
- **Internet: ON.**
- **GPU: REQUIRED** (T4 is fine).

Resume-safe. The caption is capped to ~75 CLIP tokens so CLIP-based models (SD1.5/SDXL) receive the *full* prompt instead of a silently truncated one - keeping the six prompts consistent.

In [ ]:
import sys, subprocess
def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
pip("huggingface_hub>=0.23", "transformers>=4.40", "timm", "einops",
    "accelerate", "pyarrow", "pillow")
print("deps installed")

In [ ]:
REPO_ID = "Shanmuk4622/ai-image-detection-dataset"   # <--- SAME as NB00

import os
from huggingface_hub import hf_hub_download
def load_token():
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("HF_TOKEN")
        if t: return t.strip()
    except Exception:
        pass
    for k in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        if os.environ.get(k): return os.environ[k].strip()
    import getpass
    return getpass.getpass("HF write token: ").strip()
TOKEN = load_token()

lib = hf_hub_download(REPO_ID, "ai_detect_common.py", repo_type="dataset", token=TOKEN)
import importlib.util
spec = importlib.util.spec_from_file_location("ai_detect_common", lib)
C = importlib.util.module_from_spec(spec); spec.loader.exec_module(C)
cfg = C.read_config(REPO_ID, TOKEN)
print("library + config loaded.")

## Load Florence-2-large
Florence-2's remote code tries to import `flash_attn`, which is painful to install on Kaggle. The patch below removes that import so it falls back to standard attention. We also load a CLIP tokenizer purely to count/cap tokens.

In [ ]:
import torch, re
from transformers import AutoProcessor, AutoModelForCausalLM, CLIPTokenizer

device, dtype = C.get_device_dtype()
assert device == "cuda", "Turn the GPU ON for NB02."

# --- flash-attn workaround for Florence-2 remote code ---
from unittest.mock import patch
from transformers.dynamic_module_utils import get_imports
def _no_flash(filename):
    return [x for x in get_imports(filename) if x != "flash_attn"]

MODEL = cfg["caption_model"]
with patch("transformers.dynamic_module_utils.get_imports", _no_flash):
    flo = AutoModelForCausalLM.from_pretrained(MODEL, trust_remote_code=True, torch_dtype=dtype).to(device).eval()
proc = AutoProcessor.from_pretrained(MODEL, trust_remote_code=True)
clip_tok = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
TASK = cfg["caption_task"]
MAXTOK = cfg.get("caption_max_tokens", 75)
print("Florence-2 loaded on", device)

@torch.no_grad()
def florence_caption(pil):
    inp = proc(text=TASK, images=pil, return_tensors="pt")
    input_ids = inp["input_ids"].to(device)                 # keep ids as long
    pixel_values = inp["pixel_values"].to(device, dtype)    # cast only pixels
    gen = flo.generate(input_ids=input_ids, pixel_values=pixel_values,
                       max_new_tokens=512, num_beams=3, do_sample=False)
    txt = proc.batch_decode(gen, skip_special_tokens=False)[0]
    parsed = proc.post_process_generation(txt, task=TASK, image_size=(pil.width, pil.height))
    return (parsed.get(TASK, "") or "").strip()

_LEADINS = ("the image shows ", "the image depicts ", "this image shows ",
            "this image depicts ", "the image is ", "the photo shows ",
            "an image of ", "a photo of ", "the image captures ", "this is ")
def enhance_caption(raw, max_tokens=MAXTOK):
    s = re.sub(r"\s+", " ", (raw or "").strip())
    low = s.lower()
    for p in _LEADINS:
        if low.startswith(p):
            s = s[len(p):]
            s = (s[:1].upper() + s[1:]) if s else s
            break
    if not s:
        s = "a photograph"
    ids = clip_tok(s, add_special_tokens=False)["input_ids"]
    if len(ids) > max_tokens:                                # subject is first, so trailing detail drops
        s = clip_tok.decode(ids[:max_tokens]).strip()
        ids = clip_tok(s, add_special_tokens=False)["input_ids"]
    return s, len(ids)

## Caption every real image (resume-safe)
We read the real shards, skip any id already captioned, and append new captions in batched 20-minute commits.

In [ ]:
api = C.HfApi(token=TOKEN)
done = C.reconcile_ids(REPO_ID, "captions/", TOKEN)
print("already captioned:", len(done))

cwriter = C.ShardWriter(api, REPO_ID, "captions/", schema=C.CAPTION_SCHEMA,
                        commit_interval=cfg["commit_interval_seconds"],
                        max_rows=cfg["batch_size"], token=TOKEN)
real_shards = C.list_shards(REPO_ID, "real/", TOKEN)
processed = 0
try:
    for f in real_shards:
        local = C.hf_hub_download(REPO_ID, f, repo_type="dataset", token=TOKEN)
        t = C.pq.read_table(local, columns=["image_id", "image"])
        iids = t.column("image_id").to_pylist()
        imgs = t.column("image")
        for k, iid in enumerate(iids):
            if iid in done:
                continue
            pil = C.decode_png(imgs[k].as_py())
            raw = florence_caption(pil)
            cap, ntok = enhance_caption(raw)
            cwriter.add(dict(source_real_id=iid, caption=cap, raw_caption=raw,
                             caption_model=cfg["caption_model"], caption_task=TASK,
                             n_tokens=ntok, created_utc=C.now_utc()))
            done.add(iid); processed += 1
            if processed % 200 == 0:
                print(f"  captioned {processed} this run (total ~{len(done)})")
            cwriter.maybe_flush()
finally:
    cwriter.close()
print("captioning upload complete; this run:", processed)

## Verifier
Confirms every real image has exactly one non-empty caption and none exceed the CLIP limit.

In [ ]:
from collections import Counter
real_ids = set()
for f in C.list_shards(REPO_ID, "real/", TOKEN):
    local = C.hf_hub_download(REPO_ID, f, repo_type="dataset", token=TOKEN)
    real_ids.update(C.pq.read_table(local, columns=["image_id"]).column("image_id").to_pylist())

cap_ids, empties, toolong = [], 0, 0
for f in C.list_shards(REPO_ID, "captions/", TOKEN):
    local = C.hf_hub_download(REPO_ID, f, repo_type="dataset", token=TOKEN)
    t = C.pq.read_table(local, columns=["source_real_id", "caption", "n_tokens"])
    cap_ids += t.column("source_real_id").to_pylist()
    empties += sum(1 for c in t.column("caption").to_pylist() if not c or not str(c).strip())
    toolong += sum(1 for n in t.column("n_tokens").to_pylist() if n and n > 77)

dups = [k for k, v in Counter(cap_ids).items() if v > 1]
missing = real_ids - set(cap_ids)
print(f"real={len(real_ids)}  captioned={len(set(cap_ids))}  missing={len(missing)}  "
      f"dups={len(dups)}  empty={empties}  over_77_tokens={toolong}")
print("RESULT:", "PASS" if (not missing and not dups and not empties and not toolong) else "CHECK ABOVE")